In [ ]:
from pathlib import Path

import pandas as pd

from ras_commander import (
    CalibrationPoint,
    HdfResultsPlan,
    RasCalibrate,
    RasCmdr,
    extract_steady_profile_observations,
    init_ras_project,
    make_steady_profile_calibration_points,
    make_xsec_mannings_apply_fn,
    ras,
)


## Development Mode

When running this notebook from a source checkout instead of an installed package, start Jupyter from the repository root or add the repository root to `PYTHONPATH` before launching Jupyter.

# Steady Flow Calibration Helper Scaffold

This lightweight workflow shows the first-class steady-profile calibration path for the HEC-RAS steady-flow calibration workshop shape. It uses observed WSE rows keyed by river, reach, station, and profile, then evaluates modeled WSE with `HdfResultsPlan.get_steady_wse()` through `RasCalibrate`.

In [ ]:
PROJECT_PATH = Path(r"working/steady_flow_calibration_project")
RAS_EXE = None
PLAN_NUMBER = "01"

if PROJECT_PATH.exists():
    init_ras_project(PROJECT_PATH, RAS_EXE)
    print(ras.plan_df[["plan_number", "Plan Title", "HDF_Results_Path"]])
else:
    print(f"Set PROJECT_PATH to a computed steady-flow project folder: {PROJECT_PATH}")


## Observed Profiles

Observed events or profiles are ordinary rows. Profile names must match the steady profile names in the computed plan HDF.

In [ ]:
observed_wse = pd.DataFrame(
    [
        {
            "name": "PF1 upstream",
            "river": "River A",
            "reach": "Reach 1",
            "station": "1000",
            "profile": "PF 1",
            "observed": 102.35,
        },
        {
            "name": "PF1 downstream",
            "river": "River A",
            "reach": "Reach 1",
            "station": "500",
            "profile": "PF 1",
            "observed": 98.80,
        },
        {
            "name": "PF2 upstream",
            "river": "River A",
            "reach": "Reach 1",
            "station": "1000",
            "profile": "PF 2",
            "observed": 104.10,
        },
    ]
)

calibration_points = make_steady_profile_calibration_points(
    observed_wse,
    name_column="name",
    station_tolerance=0.05,
)
calibration_points


## HDF Extraction Check

Use this when a computed plan HDF already exists. The same helper can also extract synthetic observations from a baseline run.

In [ ]:
if PROJECT_PATH.exists():
    plan_row = ras.plan_df.loc[ras.plan_df["plan_number"].astype(str).str.zfill(2) == PLAN_NUMBER].iloc[0]
    plan_hdf = Path(plan_row["HDF_Results_Path"])
    print(HdfResultsPlan.get_steady_profile_names(plan_hdf))
    print(extract_steady_profile_observations(plan_hdf).head())
    print(RasCalibrate.extract_modeled(calibration_points[0], plan_hdf))


## Manning's n Apply Function

The apply function clones the plan geometry before editing cross-section Manning's n values, so each calibration run gets its own geometry file.

In [ ]:
apply_fn = make_xsec_mannings_apply_fn(
    {
        "n_channel": {
            "river": "River A",
            "reach": "Reach 1",
            "station": "1000",
            "subsection": "Channel",
        }
    }
)

parameter_grid = {"n_channel": [0.035, 0.040, 0.045]}


In [ ]:
if PROJECT_PATH.exists():
    results = RasCalibrate.grid_search(
        template_plan=PLAN_NUMBER,
        parameters=parameter_grid,
        apply_fn=apply_fn,
        calibration_points=calibration_points,
        metric="rmse",
        suffix="steady_cal",
        max_workers=1,
        num_cores=2,
    )
    display(results[["plan_number", "n_channel", "overall_objective", "n_points_scored"]])
